In [1]:
import sys
import inspect

sys.path.append("..")

from cryoevap.storage_tanks import Tank
from cryoevap.cryogens import Cryogen

print("Archivo Tank cargado:")
print(inspect.getfile(Tank))

Archivo Tank cargado:
c:\Users\fabia\IPRE\CryoEvap\notebooks\..\cryoevap\storage_tanks\tank.py


In [2]:
import numpy as np

# Condiciones de prueba
P = 101_300.0       # Pa
T_air = 293.15      # K
V_tank = 4.89       # m³
d_i = 1.676         # m
d_o = d_i
LF_initial = 0.30
q_target = 0.50     # W/m²
eta_w = 0.97

# Propiedades del hidrógeno
hydrogen_test = Cryogen(name="hydrogen")
hydrogen_test.set_coolprops(P)

# U equivalente al flux objetivo inicial
U_equivalent = q_target / (T_air - hydrogen_test.T_sat)

# Tanque vertical con tapas planas
vertical_test = Tank(
    d_i=d_i,
    d_o=d_o,
    V=V_tank,
    vapour_geometry="cylindrical",
    liquid_geometry="cylindrical",
    LF=LF_initial,
    head_type="flat"
)

vertical_test.cryogen = hydrogen_test

vertical_test.set_HeatTransProps(
    U_L=U_equivalent,
    U_V=U_equivalent,
    T_air=T_air,
    Q_b_fixed=None,
    Q_roof=0.0,
    eta_w=eta_w
)

# Malla sencilla para esta prueba
vertical_test.z_grid = np.linspace(0.0, 1.0, 31)

# Guardar resultados cada 10 segundos
vertical_test.time_interval = 10

# Simulación corta de 60 segundos
vertical_test.evaporate(60)

print("Simulación exitosa:", vertical_test.sol.success)
print("Mensaje:", vertical_test.sol.message)
print("LF inicial:", vertical_test.data["LF"][0])
print("LF final:", vertical_test.data["LF"][-1])
print(
    "Tv_avg finita:",
    np.all(np.isfinite(vertical_test.data["Tv_avg"]))
)

Simulación exitosa: True
Mensaje: The solver successfully reached the end of the integration interval.
LF inicial: 0.3
LF final: 0.29999695641049656
Tv_avg finita: True


In [3]:
# Criógeno independiente para la prueba horizontal
hydrogen_horizontal_test = Cryogen(name="hydrogen")
hydrogen_horizontal_test.set_coolprops(P)

horizontal_test = Tank(
    d_i=d_i,
    d_o=d_o,
    V=V_tank,
    vapour_geometry="horizontal",
    liquid_geometry="horizontal",
    LF=LF_initial,
    L=2.217,
    head_type="flat"
)

horizontal_test.cryogen = hydrogen_horizontal_test

horizontal_test.set_HeatTransProps(
    U_L=U_equivalent,
    U_V=U_equivalent,
    T_air=T_air,
    Q_b_fixed=0.0,
    Q_roof=0.0,
    eta_w=eta_w
)

horizontal_test.z_grid = np.linspace(0.0, 1.0, 31)
horizontal_test.time_interval = 10

horizontal_test.evaporate(60)

print("Simulación exitosa:", horizontal_test.sol.success)
print("Mensaje:", horizontal_test.sol.message)

print("\nGeometría inicial:")
print("LF inicial:", horizontal_test.data["LF"][0])
print("z inicial:", horizontal_test.data["z"][0])
print("A_T inicial:", horizontal_test.data["A_T"][0])

print("\nGeometría final:")
print("LF final:", horizontal_test.data["LF"][-1])
print("z final:", horizontal_test.data["z"][-1])
print("A_T final:", horizontal_test.data["A_T"][-1])

print("\nComprobaciones numéricas:")
print(
    "Tv_avg finita:",
    np.all(np.isfinite(horizontal_test.data["Tv_avg"]))
)
print(
    "A_T finita y positiva:",
    np.all(np.isfinite(horizontal_test.data["A_T"]))
    and np.all(horizontal_test.data["A_T"] > 0.0)
)

Simulación exitosa: True
Mensaje: The solver successfully reached the end of the integration interval.

Geometría inicial:
LF inicial: 0.3
z inicial: 0.5700076021768731
A_T inicial: 3.5205627021471657

Geometría final:
LF final: 0.29999695450507663
z final: 0.570003372033903
A_T final: 3.5205563713174466

Comprobaciones numéricas:
Tv_avg finita: True
A_T finita y positiva: True


In [4]:
from scipy.integrate import simpson

R = d_i / 2.0
L_test = 2.217

# Volúmenes geométricos exactos
V_cylinder = np.pi * R**2 * L_test
V_flat = V_cylinder
V_hemispherical = V_cylinder + (4.0 / 3.0) * np.pi * R**3


def check_horizontal_weight(head_type, tank_volume):

    tank = Tank(
        d_i=d_i,
        d_o=d_o,
        V=tank_volume,
        vapour_geometry="horizontal",
        liquid_geometry="horizontal",
        LF=LF_initial,
        L=L_test,
        head_type=head_type
    )

    # Alturas físicas desde la interfase hasta el techo
    h_grid = np.linspace(tank.z, tank.d_i, 10_001)

    half_chord = np.sqrt(
        np.maximum(
            tank.d_i * h_grid - h_grid**2,
            0.0
        )
    )

    weight_cylinder = 2.0 * tank.L * half_chord

    if head_type == "flat":

        weight = weight_cylinder

    elif head_type == "hemispherical":

        weight_heads = np.pi * half_chord**2
        weight = weight_cylinder + weight_heads

    # Volumen de vapor obtenido integrando weight
    V_vapour_integrated = simpson(weight, x=h_grid)

    # Volumen esperado según el nivel líquido
    V_vapour_expected = tank.V * (1.0 - tank.LF)

    print(f"\nTapas: {head_type}")
    print(f"z inicial                  = {tank.z:.10f} m")
    print(f"A_T desde Tank             = {tank.A_T:.10f} m²")
    print(f"weight en la interfase     = {weight[0]:.10f} m²")
    print(f"Volumen vapor esperado     = {V_vapour_expected:.10f} m³")
    print(f"Volumen integrado          = {V_vapour_integrated:.10f} m³")
    print(
        f"Error absoluto             = "
        f"{abs(V_vapour_integrated - V_vapour_expected):.3e} m³"
    )

    assert np.isclose(
        weight[0],
        tank.A_T,
        rtol=1e-10,
        atol=1e-10
    )

    assert abs(
        V_vapour_integrated - V_vapour_expected
    ) < 1e-6


check_horizontal_weight(
    head_type="flat",
    tank_volume=V_flat
)

check_horizontal_weight(
    head_type="hemispherical",
    tank_volume=V_hemispherical
)

print("\nTodas las comprobaciones de weight fueron aprobadas.")


Tapas: flat
z inicial                  = 0.5700985148 m
A_T desde Tank             = 3.5206987350 m²
weight en la interfase     = 3.5206987350 m²
Volumen vapor esperado     = 3.4237468294 m³
Volumen integrado          = 3.4237462875 m³
Error absoluto             = 5.420e-07 m³

Tapas: hemispherical
z inicial                  = 0.5842013358 m
A_T desde Tank             = 5.5449847323 m²
weight en la interfase     = 5.5449847323 m²
Volumen vapor esperado     = 5.1492616952 m³
Volumen integrado          = 5.1492611636 m³
Error absoluto             = 5.316e-07 m³

Todas las comprobaciones de weight fueron aprobadas.


In [5]:
# Resultados reconstruidos
Tv_avg = np.asarray(horizontal_test.data["Tv_avg"])
rho_V_avg = np.asarray(horizontal_test.data["rho_V_avg"])

# Perfil de temperatura en el último instante
T_v_final = horizontal_test.sol.y[1:, -1]

# Altura líquida final
z_final = horizontal_test.data["z"][-1]

# Alturas físicas de los nodos de vapor
h_grid_final = (
    horizontal_test.z_grid
    * (horizontal_test.d_i - z_final)
    + z_final
)

half_chord_final = np.sqrt(
    np.maximum(
        horizontal_test.d_i * h_grid_final
        - h_grid_final**2,
        0.0
    )
)

# Tapas planas: solamente contribuye el cilindro
weight_final = (
    2.0
    * horizontal_test.L
    * half_chord_final
)

# Cálculo manual de Tv_avg final
Tv_avg_manual = (
    simpson(
        T_v_final * weight_final,
        x=horizontal_test.z_grid
    )
    / simpson(
        weight_final,
        x=horizontal_test.z_grid
    )
)

print("Rango de Tv_avg:")
print(f"mínimo = {np.min(Tv_avg):.10f} K")
print(f"máximo = {np.max(Tv_avg):.10f} K")

print("\nRango de rho_V_avg:")
print(f"mínimo = {np.min(rho_V_avg):.10f} kg/m³")
print(f"máximo = {np.max(rho_V_avg):.10f} kg/m³")

print("\nPropiedades promedio actuales:")
print(f"k_V_avg  = {horizontal_test.cryogen.k_V_avg}")
print(f"cp_V_avg = {horizontal_test.cryogen.cp_V_avg}")

print("\nComprobación de Tv_avg final:")
print(f"Tv_avg almacenada = {Tv_avg[-1]:.12f} K")
print(f"Tv_avg manual     = {Tv_avg_manual:.12f} K")
print(
    f"Diferencia        = "
    f"{abs(Tv_avg[-1] - Tv_avg_manual):.3e} K"
)

assert np.all(np.isfinite(Tv_avg))
assert np.all(np.isfinite(rho_V_avg))
assert np.all(rho_V_avg > 0.0)

assert np.isfinite(horizontal_test.cryogen.k_V_avg)
assert horizontal_test.cryogen.k_V_avg > 0.0

assert np.isfinite(horizontal_test.cryogen.cp_V_avg)
assert horizontal_test.cryogen.cp_V_avg > 0.0

assert np.isclose(
    Tv_avg[-1],
    Tv_avg_manual,
    rtol=1e-10,
    atol=1e-10
)

print("\nPromedios físicos y numéricos aprobados.")

Rango de Tv_avg:
mínimo = 20.3680675984 K
máximo = 20.3682357645 K

Rango de rho_V_avg:
mínimo = 1.3318584314 kg/m³
máximo = 1.3318726469 kg/m³

Propiedades promedio actuales:
k_V_avg  = 0.017450503578876442
cp_V_avg = 12036.111331872398

Comprobación de Tv_avg final:
Tv_avg almacenada = 20.368235764488 K
Tv_avg manual     = 20.368235764488 K
Diferencia        = 0.000e+00 K

Promedios físicos y numéricos aprobados.


In [6]:
z_series = np.asarray(horizontal_test.data["z"])
Tv_avg = np.asarray(horizontal_test.data["Tv_avg"])
Q_L_stored = np.asarray(horizontal_test.data["Q_L"])
Q_V_stored = np.asarray(horizontal_test.data["Q_V"])

R_o = horizontal_test.d_o / 2.0
wall_thickness = (horizontal_test.d_o - horizontal_test.d_i) / 2.0
h_o = np.clip(z_series + wall_thickness, 0.0, 2.0 * R_o)
theta = np.arccos(np.clip((R_o - h_o) / R_o, -1.0, 1.0))
sqrt_term = np.sqrt(np.maximum(2.0 * R_o * h_o - h_o**2, 0.0))

A_lateral_wet = horizontal_test.d_o * horizontal_test.L * theta
A_one_head_wet = R_o**2 * theta - (R_o - h_o) * sqrt_term
A_heads_wet = 2.0 * A_one_head_wet
A_wet_check = A_lateral_wet + A_heads_wet

A_total_check = np.pi * horizontal_test.d_o * horizontal_test.L + 2.0 * np.pi * R_o**2
A_vapour_check = A_total_check - A_wet_check

Q_L_check = horizontal_test.U_L * A_wet_check * (horizontal_test.T_air - horizontal_test.cryogen.T_sat)
Q_V_check = horizontal_test.U_V * A_vapour_check * (horizontal_test.T_air - Tv_avg)

print("Áreas iniciales:")
print(f"A_wet inicial       = {A_wet_check[0]:.10f} m²")
print(f"A_vapour inicial    = {A_vapour_check[0]:.10f} m²")
print(f"A_total             = {A_total_check:.10f} m²")
print(f"Suma de áreas       = {(A_wet_check[0] + A_vapour_check[0]):.10f} m²")

print("\nErrores máximos:")
print(f"Error máximo Q_L    = {np.max(np.abs(Q_L_stored - Q_L_check)):.3e} W")
print(f"Error máximo Q_V    = {np.max(np.abs(Q_V_stored - Q_V_check)):.3e} W")

assert np.all(np.isfinite(Q_L_stored))
assert np.all(np.isfinite(Q_V_stored))
assert np.all(A_wet_check > 0.0)
assert np.all(A_vapour_check > 0.0)
assert np.allclose(Q_L_stored, Q_L_check, rtol=1e-12, atol=1e-12)
assert np.allclose(Q_V_stored, Q_V_check, rtol=1e-12, atol=1e-12)

print("\nQ_L y Q_V horizontales aprobados.")

Áreas iniciales:
A_wet inicial       = 5.9504792740 m²
A_vapour inicial    = 10.1350405991 m²
A_total             = 16.0855198731 m²
Suma de áreas       = 16.0855198731 m²

Errores máximos:
Error máximo Q_L    = 0.000e+00 W
Error máximo Q_V    = 0.000e+00 W

Q_L y Q_V horizontales aprobados.


In [7]:
B_L_stored = np.asarray(vertical_test.data["B_L"]) * 3600.0
dV_L = np.asarray(vertical_test.data["dV_L"])
B_L_from_volume = -vertical_test.cryogen.rho_L * dV_L * 3600.0

error_B_L = np.abs(B_L_stored[1:] - B_L_from_volume[1:])

print("Evaporación:")
print(f"B_L almacenado inicial        = {B_L_stored[0]:.10f} kg/h")
print(f"B_L almacenado final          = {B_L_stored[-1]:.10f} kg/h")
print(f"-rho_L dV_L/dt final          = {B_L_from_volume[-1]:.10f} kg/h")
print(f"Error máximo sin t=0          = {np.max(error_B_L):.3e} kg/h")

Q_L_initial = vertical_test.data["Q_L"][0]
Q_b = vertical_test.Q_b

print("\nCalores reconstruidos:")
print(f"Q_L lateral inicial           = {Q_L_initial:.10f} W")
print(f"Q_b del fondo                 = {Q_b:.10f} W")
print(f"Q_L + Q_b                     = {(Q_L_initial + Q_b):.10f} W")

print("\nComprobaciones:")
print("B_L finito:", np.all(np.isfinite(B_L_stored)))
print("Coincidencia al final:", np.isclose(B_L_stored[-1], B_L_from_volume[-1], rtol=1e-5, atol=1e-7))

Evaporación:
B_L almacenado inicial        = 0.0632674109 kg/h
B_L almacenado final          = 0.0632685586 kg/h
-rho_L dV_L/dt final          = 0.0632682566 kg/h
Error máximo sin t=0          = 7.466e-07 kg/h

Calores reconstruidos:
Q_L lateral inicial           = 1.7505966587 W
Q_b del fondo                 = 1.1030822957 W
Q_L + Q_b                     = 2.8536789544 W

Comprobaciones:
B_L finito: True
Coincidencia al final: True


In [8]:
hydrogen_5h = Cryogen(name="hydrogen")
hydrogen_5h.set_coolprops(P)

vertical_5h = Tank(d_i=d_i, d_o=d_o, V=V_tank, vapour_geometry="cylindrical", liquid_geometry="cylindrical", LF=LF_initial, head_type="flat")
vertical_5h.cryogen = hydrogen_5h

vertical_5h.set_HeatTransProps(U_L=U_equivalent, U_V=U_equivalent, T_air=T_air, Q_b_fixed=None, Q_roof=0.0, eta_w=eta_w)

dz_target = 0.01
n_z = max(3, 1 + int(np.round(vertical_5h.l_V / dz_target)))

vertical_5h.z_grid = np.linspace(0.0, 1.0, n_z)
vertical_5h.time_interval = 60

vertical_5h.evaporate(5 * 3600)

time_h = vertical_5h.sol.t / 3600.0
V_L = np.asarray(vertical_5h.data["V_L"])
Tv_avg = np.asarray(vertical_5h.data["Tv_avg"])
rho_V_avg = np.asarray(vertical_5h.data["rho_V_avg"])

m_L = vertical_5h.cryogen.rho_L * V_L
m_V = rho_V_avg * (vertical_5h.V - V_L)
m_total = m_L + m_V

dM_L_dt = np.gradient(m_L, time_h, edge_order=2)
dM_V_dt = np.gradient(m_V, time_h, edge_order=2)

evaporation = -dM_L_dt
BOG_balance = -(dM_L_dt + dM_V_dt)

mass_released = simpson(BOG_balance, x=time_h)
mass_change = m_total[-1] - m_total[0]
mass_balance_error = mass_change + mass_released

print("Simulación vertical de 5 h:")
print(f"Solver exitoso              = {vertical_5h.sol.success}")
print(f"Mensaje                     = {vertical_5h.sol.message}")
print(f"Número de evaluaciones      = {vertical_5h.sol.nfev}")
print(f"Número de nodos             = {n_z}")

print("\nResultados finales:")
print(f"LF inicial                  = {vertical_5h.data['LF'][0]:.10f}")
print(f"LF final                    = {vertical_5h.data['LF'][-1]:.10f}")
print(f"Tv_avg inicial              = {Tv_avg[0]:.10f} K")
print(f"Tv_avg final                = {Tv_avg[-1]:.10f} K")
print(f"Evaporación final           = {evaporation[-1]:.10f} kg/h")
print(f"BOG final                   = {BOG_balance[-1]:.10f} kg/h")

print("\nBalance total:")
print(f"Cambio de masa              = {mass_change:.10f} kg")
print(f"Masa liberada               = {mass_released:.10f} kg")
print(f"Error de balance            = {mass_balance_error:.3e} kg")

print("\nComprobaciones numéricas:")
print("Tv_avg finita:", np.all(np.isfinite(Tv_avg)))
print("rho_V_avg positiva:", np.all(np.isfinite(rho_V_avg)) and np.all(rho_V_avg > 0.0))
print("LF positiva:", np.all(vertical_5h.data["LF"] > 0.0))

Simulación vertical de 5 h:
Solver exitoso              = True
Mensaje                     = The solver successfully reached the end of the integration interval.
Número de evaluaciones      = 1706
Número de nodos             = 156

Resultados finales:
LF inicial                  = 0.3000000000
LF final                    = 0.2990862402
Tv_avg inicial              = 20.3680782788 K
Tv_avg final                = 20.4091815749 K
Evaporación final           = 0.0633309378 kg/h
BOG final                   = 0.0642477236 kg/h

Balance total:
Cambio de masa              = -0.3224923226 kg
Masa liberada               = 0.3224920502 kg
Error de balance            = -2.723e-07 kg

Comprobaciones numéricas:
Tv_avg finita: True
rho_V_avg positiva: True
LF positiva: True
